# Artificial Neural Network (ANN) — E-Commerce Purchase Prediction

## Beginner-friendly training notebook

This notebook teaches the **complete basic flow of a neural network** using a small e-commerce dataset.

We will predict whether a customer purchases a premium product:

- `Purchased = 1` means the customer purchased.
- `Purchased = 0` means the customer did not purchase.

The notebook connects the code to the important neural-network terms:

**Inputs → Weights → Bias → Weighted Sum → Activation → Prediction → Loss → Backpropagation → Weight Update**

### Files used

- Dataset: `ecommerce_ann_dataset.csv`
- Records: **200**
- Columns: **10**, including the target column

### What participants will learn

1. How a CSV row becomes input to a neural network.
2. Why categorical values must be converted into numbers.
3. Why numerical columns are scaled.
4. How an artificial neuron performs its calculation.
5. How input, hidden and output layers are created.
6. How forward propagation produces predictions.
7. How loss measures mistakes.
8. How backpropagation and the optimizer update weights.
9. How epoch, batch and iteration are related.
10. How to evaluate and use the trained ANN.

## 1. Business problem

An e-commerce company wants to identify customers who are likely to purchase a premium product.

The model uses customer activity such as:

- age and annual income,
- number of website visits,
- average session time,
- pages viewed,
- cart value,
- discount percentage,
- membership level,
- device type.

The final output is a binary prediction:

```text
0 → Customer did not purchase
1 → Customer purchased
```

This is called a **binary classification problem** because only two output classes exist.

## 2. Neural-network flow used in this notebook

```text
CSV row
   ↓
Select input columns
   ↓
Convert categories into numbers
   ↓
Scale numerical columns
   ↓
Input layer
   ↓
Hidden layer 1: Dense(16, ReLU)
   ↓
Hidden layer 2: Dense(8, ReLU)
   ↓
Output layer: Dense(1, Sigmoid)
   ↓
Probability between 0 and 1
   ↓
Convert probability into class 0 or 1
```

During training, the model repeatedly follows this cycle:

```text
Forward pass
   ↓
Prediction
   ↓
Calculate loss
   ↓
Backpropagation
   ↓
Optimizer updates weights and biases
   ↓
Repeat for the next batch and epoch
```

## 3. Install the required libraries

Run the following command in the terminal before opening the notebook:

```bash
pip install -r requirements.txt
```

The installation cell below is commented out so that it does not reinstall packages every time.

In [ ]:
# Remove the # symbol only when the required libraries are not installed.
# !pip install -r requirements.txt

## 4. Import libraries

- **pandas** reads and processes the CSV file.
- **numpy** performs numerical operations.
- **matplotlib** creates graphs.
- **scikit-learn** splits and preprocesses the data and calculates evaluation metrics.
- **TensorFlow/Keras** builds and trains the artificial neural network.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Fix random seeds so that results are more reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

## 5. Load the e-commerce CSV

A dataframe is a table containing rows and columns.

Each row represents one customer.

Each column represents one input feature or the target.

In [ ]:
df = pd.read_csv("ecommerce_ann_dataset.csv")

print("Dataset shape:", df.shape)
df.head()

### Expected shape

The dataset contains:

```text
200 rows × 10 columns
```

Nine columns are inputs, and one column—`Purchased`—is the target.

The model will **not** receive the correct answer as an input. The target is separated before training.

In [ ]:
print("Column names:")
for column in df.columns:
    print("-", column)

## 6. Understand every dataset column

| Column | Meaning | Type |
|---|---|---|
| Customer_Age | Customer age in years | Numerical |
| Annual_Income | Approximate annual income | Numerical |
| Website_Visits | Number of recent website visits | Numerical |
| Avg_Session_Minutes | Average time spent per visit | Numerical |
| Pages_Viewed | Number of pages viewed | Numerical |
| Cart_Value | Current cart amount | Numerical |
| Discount_Percent | Discount offered | Numerical |
| Membership_Level | Basic, Silver or Gold | Categorical |
| Device_Type | Mobile, Desktop or Tablet | Categorical |
| Purchased | Correct output: 0 or 1 | Target |

A neural network accepts numbers. Therefore, text categories such as `Gold` and `Mobile` must be converted into numerical form.

In [ ]:
df.info()

## 7. Basic data-quality checks

Before training, check:

- missing values,
- duplicate records,
- target-class distribution,
- basic numerical statistics.

A model may learn misleading patterns when the data contains errors or when one class is extremely rare.

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
print(df["Purchased"].value_counts())

print("\nTarget percentages:")
print((df["Purchased"].value_counts(normalize=True) * 100).round(2))

In [ ]:
df.describe(include="all").T

## 8. Visualize the target classes

The graph shows the number of customers in each class.

A reasonably balanced dataset helps the beginner focus on ANN concepts without first dealing with advanced imbalance techniques.

In [ ]:
class_counts = df["Purchased"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(["Not Purchased (0)", "Purchased (1)"], class_counts.values)
plt.title("Target Class Distribution")
plt.ylabel("Number of Customers")
plt.show()

## 9. Separate inputs and target

### Inputs: `X`

`X` contains the information given to the network.

### Target: `y`

`y` contains the correct answer that the network must learn to predict.

```text
X → customer attributes
y → Purchased
```

In [ ]:
X = df.drop(columns=["Purchased"])
y = df["Purchased"]

print("Input shape:", X.shape)
print("Target shape:", y.shape)

X.head()

## 10. Split the data into training, validation and test sets

### Training data

Used to update weights and biases.

### Validation data

Used during training to check whether the model is improving on unseen data.

### Test data

Used only after training to measure final performance.

The split used here is approximately:

```text
Training   → 70%
Validation → 15%
Test       → 15%
```

`stratify=y` keeps the class proportions similar in every split.

In [ ]:
# First, reserve 30% for validation + test.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

# Divide the remaining 30% equally into validation and test.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Training records:", len(X_train))
print("Validation records:", len(X_val))
print("Test records:", len(X_test))

## 11. Preprocess the inputs

Two types of columns exist.

### Numerical columns

These are standardized with `StandardScaler`.

Standardization approximately converts values to:

```text
mean = 0
standard deviation = 1
```

Why is scaling required?

`Annual_Income` may contain values such as `75000`, while `Discount_Percent` may contain `10`.

Without scaling, large-value columns may dominate the learning process.

### Categorical columns

`OneHotEncoder` converts categories into separate 0/1 columns.

Example:

```text
Membership_Level = Gold

Basic  Silver  Gold
  0      0      1
```

The preprocessor is fitted only on the training data to prevent data leakage.

In [ ]:
numerical_columns = [
    "Customer_Age",
    "Annual_Income",
    "Website_Visits",
    "Avg_Session_Minutes",
    "Pages_Viewed",
    "Cart_Value",
    "Discount_Percent"
]

categorical_columns = [
    "Membership_Level",
    "Device_Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numerical_columns),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns
        )
    ]
)

# Learn preprocessing values from training data only.
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same learned transformation to validation and test data.
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

### Why did the number of columns increase?

The original input has nine columns.

Two categorical columns are expanded into multiple one-hot columns:

- Membership level becomes Basic, Gold and Silver columns.
- Device type becomes Desktop, Mobile and Tablet columns.

Therefore, the processed data contains more numerical input features than the original CSV.

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Features given to the ANN:")
for index, name in enumerate(feature_names, start=1):
    print(f"{index:2d}. {name}")

## 12. View one processed customer

After preprocessing, one customer is represented entirely by numbers.

These numbers become the **input values** received by the ANN input layer.

In [ ]:
first_customer = pd.DataFrame(
    [X_train_processed[0]],
    columns=feature_names
)

first_customer.T

# Part A — Understand one artificial neuron manually

Before building the complete network, let us perform one neuron's calculation.

A neuron performs:

\[
z = w_1x_1 + w_2x_2 + \cdots + w_nx_n + b
\]

Then it applies an activation function:

\[
a = f(z)
\]

Where:

- `x` = inputs,
- `w` = weights,
- `b` = bias,
- `z` = weighted sum,
- `f` = activation function,
- `a` = neuron output.

## 13. Manual neuron calculation

For demonstration, take the first three processed input values.

We create example weights and a bias manually.

These are **not yet learned weights**. They are used only to explain the calculation.

In [ ]:
# Take three values from one processed customer.
example_inputs = X_train_processed[0][:3]

# Example weights and bias for teaching.
example_weights = np.array([0.5, -0.2, 0.8])
example_bias = 0.1

weighted_values = example_inputs * example_weights
z = weighted_values.sum() + example_bias

# ReLU activation: negative values become 0.
relu_output = max(0, z)

print("Inputs:", example_inputs)
print("Weights:", example_weights)
print("Input × weight:", weighted_values)
print("Bias:", example_bias)
print("Weighted sum z:", z)
print("Output after ReLU:", relu_output)

### Interpretation of the manual calculation

1. Every input is multiplied by its weight.
2. Positive weights increase the weighted sum when the input is positive.
3. Negative weights can reduce the weighted sum.
4. Bias is added after the weighted inputs are summed.
5. ReLU returns zero for a negative weighted sum.
6. ReLU returns the original value for a positive weighted sum.

A complete neural network performs calculations like this in many neurons simultaneously.

# Part B — Build the complete ANN architecture

The network used in this notebook is:

```text
Input layer
    ↓
Dense hidden layer: 16 neurons + ReLU
    ↓
Dense hidden layer: 8 neurons + ReLU
    ↓
Output layer: 1 neuron + Sigmoid
```

### Why 16 and 8 neurons?

They are beginner-friendly hyperparameters. They are selected by the developer, not learned automatically.

### Why one output neuron?

This is binary classification: purchased or not purchased.

### Why sigmoid?

Sigmoid converts the final value into a probability between 0 and 1.

In [ ]:
number_of_input_features = X_train_processed.shape[1]

model = Sequential([
    Input(shape=(number_of_input_features,), name="Input_Layer"),

    Dense(
        units=16,
        activation="relu",
        name="Hidden_Layer_1"
    ),

    Dense(
        units=8,
        activation="relu",
        name="Hidden_Layer_2"
    ),

    Dense(
        units=1,
        activation="sigmoid",
        name="Output_Layer"
    )
])

model.summary()

## 14. Understand `model.summary()`

The summary displays:

### Layer

The name and type of each layer.

### Output shape

The number of values produced by that layer for every record.

### Parameters

The number of weights and biases in the layer.

For a dense layer:

\[
Parameters = (Number\ of\ inputs × Number\ of\ neurons) + Number\ of\ biases
\]

Each neuron has one bias.

Example: if a layer receives 13 inputs and contains 16 neurons:

```text
Weights = 13 × 16 = 208
Biases  = 16
Total   = 224 parameters
```

These parameters are the values learned during training.

In [ ]:
# Calculate parameter counts manually for teaching.
input_features = number_of_input_features

layer_1_parameters = (input_features * 16) + 16
layer_2_parameters = (16 * 8) + 8
output_parameters = (8 * 1) + 1
total_parameters = layer_1_parameters + layer_2_parameters + output_parameters

print("Hidden layer 1 parameters:", layer_1_parameters)
print("Hidden layer 2 parameters:", layer_2_parameters)
print("Output layer parameters:", output_parameters)
print("Total trainable parameters:", total_parameters)

## 15. Compile the ANN

Compilation defines how the model learns.

### Optimizer: Adam

The optimizer updates weights and biases using gradients calculated during backpropagation.

### Loss: Binary cross-entropy

The loss function measures how far predicted probabilities are from correct binary answers.

A lower loss generally means better predictions.

### Metric: Accuracy

Accuracy is the percentage of correctly classified records.

```text
Accuracy = Correct predictions / Total predictions
```

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## 16. Understand the complete training cycle

For every batch, Keras performs the following automatically:

### Step 1: Forward propagation

Inputs move through all layers and produce probabilities.

### Step 2: Loss calculation

Predictions are compared with correct answers.

### Step 3: Backpropagation

The error is sent backward through the network. Gradients show how each weight contributed to the error.

### Step 4: Optimizer update

Adam changes the weights and biases in a direction expected to reduce loss.

### Step 5: Repeat

The same process continues for all batches and epochs.

## 17. Epoch, batch and iteration

### Epoch

One complete pass through the full training dataset.

### Batch

A small group of training records processed together.

### Iteration

One parameter update using one batch.

For example, with 140 training records and batch size 16:

```text
Iterations per epoch = ceiling(140 / 16) = 9
```

For 30 epochs:

```text
Maximum iterations = 9 × 30 = 270
```

The actual run may stop earlier because this notebook uses early stopping.

In [ ]:
batch_size = 16
maximum_epochs = 50

iterations_per_epoch = math.ceil(len(X_train_processed) / batch_size)

print("Training records:", len(X_train_processed))
print("Batch size:", batch_size)
print("Iterations per epoch:", iterations_per_epoch)
print("Maximum possible iterations:", iterations_per_epoch * maximum_epochs)

## 18. Early stopping

Early stopping watches validation loss.

Training stops when validation loss does not improve for several epochs.

This helps reduce overfitting and restores the weights from the best validation epoch.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

## 19. Train the model

During each epoch, observe:

- `loss`: error on training data,
- `accuracy`: accuracy on training data,
- `val_loss`: error on validation data,
- `val_accuracy`: accuracy on validation data.

Healthy learning usually shows decreasing loss and improving accuracy, although small fluctuations are normal.

In [ ]:
history = model.fit(
    X_train_processed,
    y_train,
    validation_data=(X_val_processed, y_val),
    epochs=maximum_epochs,
    batch_size=batch_size,
    callbacks=[early_stopping],
    verbose=1
)

## 20. Plot training and validation loss

Loss tells us how wrong the model is.

Typical interpretation:

- Both losses decrease: the model is learning.
- Training loss decreases but validation loss increases: possible overfitting.
- Both remain high: possible underfitting.

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(8, 5))
plt.plot(history_df.index + 1, history_df["loss"], label="Training Loss")
plt.plot(history_df.index + 1, history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

## 21. Plot training and validation accuracy

Accuracy shows the percentage of correct class predictions.

Validation accuracy is more useful than training accuracy for understanding performance on unseen records during model development.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df.index + 1, history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df.index + 1, history_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()

# Part C — Evaluate the ANN

The test set has remained unseen during training.

We now use it to estimate how the model may perform on new customers.

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_processed,
    y_test,
    verbose=0
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

## 22. Probabilities and final classes

The sigmoid output is a probability.

Example:

```text
0.82 → 82% estimated probability of purchase
```

A threshold converts the probability into a class.

```text
Probability >= 0.50 → class 1
Probability <  0.50 → class 0
```

The threshold is a business decision and does not always have to be 0.50.

In [ ]:
test_probabilities = model.predict(X_test_processed, verbose=0).ravel()
test_predictions = (test_probabilities >= 0.50).astype(int)

prediction_table = X_test.reset_index(drop=True).copy()
prediction_table["Actual_Purchased"] = y_test.reset_index(drop=True)
prediction_table["Purchase_Probability"] = test_probabilities.round(4)
prediction_table["Predicted_Purchased"] = test_predictions

prediction_table.head(10)

## 23. Confusion matrix

The confusion matrix shows four outcomes:

- **True Negative:** correctly predicted no purchase.
- **False Positive:** predicted purchase, but no purchase occurred.
- **False Negative:** predicted no purchase, but purchase occurred.
- **True Positive:** correctly predicted purchase.

This gives more detail than accuracy alone.

In [ ]:
cm = confusion_matrix(y_test, test_predictions)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Purchased", "Purchased"]
)

display.plot()
plt.title("ANN Confusion Matrix")
plt.show()

## 24. Classification report

### Precision

Of all customers predicted as purchasers, how many actually purchased?

### Recall

Of all actual purchasers, how many did the model identify?

### F1-score

A combined measure of precision and recall.

### Support

The number of actual records belonging to each class.

In [ ]:
print(classification_report(
    y_test,
    test_predictions,
    target_names=["Not Purchased", "Purchased"],
    zero_division=0
))

# Part D — Inspect what the network learned

A neural network learns numerical weights and biases.

The first hidden layer connects every processed input feature to every one of its 16 neurons.

In [ ]:
first_hidden_layer = model.get_layer("Hidden_Layer_1")
learned_weights, learned_biases = first_hidden_layer.get_weights()

print("Weight matrix shape:", learned_weights.shape)
print("Bias vector shape:", learned_biases.shape)

print("\nFirst five input weights connected to neuron 1:")
print(learned_weights[:5, 0])

print("\nBias of neuron 1:")
print(learned_biases[0])

### Weight-matrix interpretation

If the weight matrix shape is `(13, 16)`:

- 13 rows represent processed input features.
- 16 columns represent neurons in the first hidden layer.
- Each cell is one learned connection weight.

The bias vector contains 16 values because every hidden neuron has one bias.

Individual ANN weights are usually not interpreted as simple business rules because later layers combine many values non-linearly.

## 25. Make a prediction for one new customer

The new record must contain the same original input columns used during training.

The preprocessor converts it into the same numerical format before the ANN receives it.

In [ ]:
new_customer = pd.DataFrame([{
    "Customer_Age": 38,
    "Annual_Income": 85000,
    "Website_Visits": 18,
    "Avg_Session_Minutes": 13.5,
    "Pages_Viewed": 35,
    "Cart_Value": 240.0,
    "Discount_Percent": 15,
    "Membership_Level": "Gold",
    "Device_Type": "Mobile"
}])

new_customer_processed = preprocessor.transform(new_customer)

purchase_probability = model.predict(
    new_customer_processed,
    verbose=0
)[0][0]

predicted_class = int(purchase_probability >= 0.50)

print(f"Purchase probability: {purchase_probability:.4f}")
print("Predicted class:", predicted_class)
print(
    "Interpretation:",
    "Likely to purchase" if predicted_class == 1
    else "Not likely to purchase"
)

# Part E — Connect every ANN term to this example

| Neural-network term | Meaning in this notebook |
|---|---|
| Input | Processed customer features |
| Weight | Importance of one connection |
| Bias | Extra trainable value for each neuron |
| Weighted sum | Input × weight values, added with bias |
| ReLU | Hidden-layer activation |
| Sigmoid | Converts output into purchase probability |
| Input layer | Receives all processed features |
| Hidden layer | Learns combinations and patterns |
| Output layer | Produces one probability |
| Forward propagation | Inputs move to the output |
| Loss | Measures prediction error |
| Backpropagation | Calculates gradients from the error |
| Optimizer | Adam updates weights and biases |
| Learning rate | Size of each optimizer update |
| Batch | Small group of customer records |
| Iteration | One update using one batch |
| Epoch | One full pass through training data |
| Validation set | Checks generalization during training |
| Test set | Final evaluation on unseen data |

# Part F — Trainer explanation script

Use this simple sequence while presenting the notebook:

1. **A CSV row is one customer.**
2. **The target tells us whether the customer purchased.**
3. **The ANN cannot process text, so categories are encoded.**
4. **Numerical values are scaled so that columns are comparable.**
5. **The processed columns enter the input layer.**
6. **Each hidden neuron multiplies inputs by weights and adds a bias.**
7. **ReLU transforms the hidden neuron's weighted sum.**
8. **The next layer receives outputs from the previous layer.**
9. **The sigmoid output gives a purchase probability.**
10. **Loss compares the probability with the correct answer.**
11. **Backpropagation calculates how the weights should change.**
12. **Adam updates the weights and biases.**
13. **This repeats for multiple batches and epochs.**
14. **The test set checks performance on previously unseen customers.**

# Final summary

A neural network does not directly understand customers or business meaning.

It receives numerical inputs and learns mathematical relationships.

```text
Customer data
    ↓
Preprocessing
    ↓
Input layer
    ↓
Weights + biases
    ↓
Hidden layers with ReLU
    ↓
Output layer with sigmoid
    ↓
Purchase probability
```

During training:

```text
Predict
  ↓
Measure loss
  ↓
Backpropagate error
  ↓
Update weights
  ↓
Repeat
```

This ANN foundation prepares participants for image processing.

In a CNN, image pixels become inputs, and convolution layers learn visual features such as edges, textures, shapes and object parts. The underlying learning process—forward propagation, loss, backpropagation and weight updates—remains the same.